# 06 — Proposed Retrieval: BGE-small + ChromaDB
**Project:** Semantic Book Recommender — IT4142 HUST  
**Input:**
- `data/processed/books_with_emotions.csv`

**Output:**
- `data/chroma_db/` — persistent ChromaDB collection containing embeddings and metadata

Pipeline:
1. Load data with emotion scores
2. Encode descriptions with `BAAI/bge-small-en-v1.5`
3. Build ChromaDB collection (upsert with metadata, including emotions)
4. Define semantic search function & smoke test

## 0. Setup

> **RAM requirement:**  
> - `BAAI/bge-small-en-v1.5` (33M params) — needs ~500MB RAM, works on CPU  
> - If RAM < 4GB: use `sentence-transformers/all-MiniLM-L6-v2` instead (see cell below)

In [1]:
# pip install sentence-transformers chromadb
import pandas as pd
import numpy as np
import json, time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

DATA_PATH   = Path('data/processed/books_with_emotions.csv')
CHROMA_PATH = Path('data/chroma_db')
CHROMA_PATH.mkdir(parents=True, exist_ok=True)

print('Setup OK')

Setup OK


## 1. Load Data

In [2]:
df = pd.read_csv(DATA_PATH)
print(f'Books loaded: {len(df):,}')

df[['isbn13', 'title', 'categories']].head(3)

Books loaded: 11,606


,isbn13,title,categories
0,9780002188319,The Complete Book of the World Cup,Soccer
1,9780002240802,The Last Thing He Wanted,Adventure stories
2,9780002246484,Royal Assassin,Assassins


## 2. Load Embedding Model

In [3]:
# Primary model: bge-small-en-v1.5 (recommended)
# Fallback:      all-MiniLM-L6-v2  (lighter, ~80MB)
MODEL_NAME = 'BAAI/bge-small-en-v1.5'
# MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'  # uncomment if low RAM

print(f'Loading model: {MODEL_NAME}')
t0 = time.time()
embed_model = SentenceTransformer(MODEL_NAME)
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'Embedding dim: {embed_model.get_sentence_embedding_dimension()}')

Loading model: BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded in 6.1s
Embedding dim: 384


## 3. Encode Book Descriptions

> **BGE prefix:** BGE models perform better when queries are prefixed with `"Represent this sentence: "`.  
> For **documents** (descriptions), no prefix needed.  
> For **queries**, we add the prefix in the search function.

In [4]:
descriptions = df['description'].tolist()

print(f'Encoding {len(descriptions):,} descriptions...')
t0 = time.time()

embeddings = embed_model.encode(
    descriptions,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2 normalize → cosine sim = dot product
)

elapsed = time.time() - t0
print(f'Encoded in {elapsed:.1f}s  ({elapsed/len(descriptions)*1000:.1f} ms/book)')
print(f'Embeddings shape: {embeddings.shape}')

Encoding 11,606 descriptions...


Batches:   0%|          | 0/182 [00:00<?, ?it/s]

Encoded in 140.5s  (12.1 ms/book)
Embeddings shape: (11606, 384)


## 4. Build ChromaDB Collection

In [5]:
client = chromadb.PersistentClient(
    path=str(CHROMA_PATH),
    settings=Settings(anonymized_telemetry=False),
)

# Delete collection if exists (clean rebuild)
try:
    client.delete_collection('books')
    print('Deleted existing collection')
except Exception:
    pass

collection = client.create_collection(
    name='books',
    metadata={'hnsw:space': 'cosine'},
)
print(f'Created collection: {collection.name}')

Created collection: books


In [6]:
# Prepare metadata — ChromaDB requires scalar values only (no list/dict)
def safe_str(val, default='Unknown'):
    return str(val) if pd.notna(val) else default

def safe_float(val, default=0.0):
    try:
        return float(val)
    except Exception:
        return default

metadatas = [
    {
        'title'          : safe_str(row['title']),
        'authors'        : safe_str(row['authors']),
        'categories'     : safe_str(row['categories']),
        'thumbnail'      : safe_str(row['thumbnail']),
        'average_rating' : safe_float(row['average_rating']),
        'published_year' : int(row['published_year']) if pd.notna(row.get('published_year')) else 0,
        # top_emotions is now read directly from the books_with_emotions.csv dataset!
        'top_emotions'   : safe_str(row.get('top_emotions', '')),
    }
    for _, row in df.iterrows()
]

ids = df['isbn13'].astype(str).tolist()

print(f'Prepared {len(metadatas):,} metadata records')

Prepared 11,606 metadata records


In [7]:
# Upsert in batches of 500 (ChromaDB recommended batch size)
BATCH_SIZE = 500
n = len(df)
t0 = time.time()

for start in range(0, n, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n)
    collection.upsert(
        ids=ids[start:end],
        embeddings=embeddings[start:end].tolist(),
        documents=descriptions[start:end],
        metadatas=metadatas[start:end],
    )
    print(f'  Upserted [{start}:{end}]')

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s')
print(f'Collection count: {collection.count():,}')

  Upserted [0:500]
  Upserted [500:1000]
  Upserted [1000:1500]
  Upserted [1500:2000]
  Upserted [2000:2500]


  Upserted [2500:3000]
  Upserted [3000:3500]
  Upserted [3500:4000]
  Upserted [4000:4500]
  Upserted [4500:5000]


  Upserted [5000:5500]
  Upserted [5500:6000]
  Upserted [6000:6500]
  Upserted [6500:7000]
  Upserted [7000:7500]


  Upserted [7500:8000]
  Upserted [8000:8500]
  Upserted [8500:9000]
  Upserted [9000:9500]


  Upserted [9500:10000]
  Upserted [10000:10500]
  Upserted [10500:11000]
  Upserted [11000:11500]
  Upserted [11500:11606]

Done in 7.1s
Collection count: 11,606


## 5. Semantic Search Function

In [8]:
# BGE prefix for queries (improves retrieval quality)
BGE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

def search_semantic(
    query: str,
    top_k: int = 10,
    filter_emotions: list = None,
) -> pd.DataFrame:
    """
    Semantic search using BGE-small + ChromaDB.

    Args:
        query           : natural language query string
        top_k           : number of results to return
        filter_emotions : list of emotion labels to filter by (requires notebook 06 to have run)
    Returns:
        DataFrame with columns: isbn13, title, authors, categories, average_rating, score
    """
    # Encode query with BGE prefix
    q_emb = embed_model.encode(
        [BGE_QUERY_PREFIX + query],
        normalize_embeddings=True,
    )

    # Build optional where filter for ChromaDB
    where = None
    if filter_emotions:
        # top_emotions is stored as comma-separated string
        # ChromaDB where syntax supports $contains for string fields
        if len(filter_emotions) == 1:
            where = {'top_emotions': {'$contains': filter_emotions[0]}}
        else:
            where = {
                '$or': [
                    {'top_emotions': {'$contains': e}}
                    for e in filter_emotions
                ]
            }

    query_kwargs = dict(
        query_embeddings=q_emb.tolist(),
        n_results=top_k,
        include=['metadatas', 'distances', 'documents'],
    )
    if where:
        query_kwargs['where'] = where

    results = collection.query(**query_kwargs)

    # Parse ChromaDB response into DataFrame
    rows = []
    for i, (meta, dist, doc) in enumerate(zip(
        results['metadatas'][0],
        results['distances'][0],
        results['documents'][0],
    )):
        rows.append({
            'isbn13'         : results['ids'][0][i],
            'title'          : meta.get('title', ''),
            'authors'        : meta.get('authors', ''),
            'categories'     : meta.get('categories', ''),
            'average_rating' : meta.get('average_rating', 0.0),
            'top_emotions'   : meta.get('top_emotions', ''),
            'score'          : round(1 - dist, 4),  # cosine distance → similarity
        })

    return pd.DataFrame(rows)


# Smoke test
test_res = search_semantic('a thriller set in Japan with a detective protagonist', top_k=5)
print('Semantic smoke test:')
test_res[['title', 'authors', 'categories', 'score']]

Semantic smoke test:


,title,authors,categories,score
0,Inspector Imanishi Investigates,Seicho Matsumoto,Fiction,0.7468
1,Salvation of a Saint,Keigo Higashino,Fiction,0.7440
2,Ronin: Have Sword- Will Travel,Kristie Lynn Higgins,Unknown,0.7362
3,The Decagon House Murders,Yukito Ayatsuji,Fiction,0.7300
4,The Devil's Disciple,Shiro Hamao,Fiction,0.7296
